In [1]:
import pandas as pd
import os
from pathlib import Path

In [5]:
def process_csv_file(csv_path):
    """
    Loads a CSV, processes the specified columns based on 'score',
    and saves the result as an Excel file with multiple sheets.
    """
    try:
        # Load the CSV
        df = pd.read_csv(csv_path)

        # Ensure 'score' column is numeric for comparison
        # Coerce errors will turn non-numeric values into NaN, which fillna(0) handles.
        df['score'] = pd.to_numeric(df['score'], errors='coerce').fillna(0).astype(int)

        # Define the mask for rows where score is 0
        score_0_mask = df['score'] == 0

        # Replace content for rows where score is 0
        # We use .loc[mask, column_name] to safely modify the DataFrame
        df.loc[score_0_mask, 'question'] = df.loc[score_0_mask, 'How can I improve the question']
        df.loc[score_0_mask, 'option']   = df.loc[score_0_mask, 'How can I improve the Options']
        df.loc[score_0_mask, 'answer']   = df.loc[score_0_mask, 'Correct Answer']

        # Define the output Excel file path
        # It will be in the same folder as the CSV, with a "Processed_" prefix
        excel_path = csv_path.with_name(f"Filtered_{csv_path.stem}.xlsx")
        
        # Get unique sheet names from the 'sheet' column
        if 'sheet' not in df.columns:
            print(f"  [ERROR] 'sheet' column not found in {csv_path}. Skipping Excel creation.")
            return

        unique_sheets = df['sheet'].unique()

        # Write to Excel, with one sheet per unique "sheet" value
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            for sheet_name in unique_sheets:
                # Filter the dataframe for the current sheet
                sheet_df = df[df['sheet'] == sheet_name].copy()
                
                # --- Optional ---
                # Drop the "helper" columns to clean up the final Excel file
                # If you want to KEEP them, just delete the line below
                sheet_df.drop(columns=['What\'s wrong in Question', 'How can I improve the question', 'How can I improve the Options', 'Correct Answer'], inplace=True, errors='ignore')

                # Write this subset to a specific sheet
                sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
        
        print(f"  [SUCCESS] Processed and saved to {excel_path.name}")

    except pd.errors.EmptyDataError:
        print(f"  [SKIPPED] {csv_path.name} is empty.")
    except FileNotFoundError:
        print(f"  [ERROR] File not found: {csv_path}")
    except Exception as e:
        print(f"  [ERROR] Failed to process {csv_path.name}: {e}")



def main():
    """
    Main function to find and process all relevant CSV files.
    """
    # This script assumes it is placed in the root "COUNTRY" folder
    # (the one containing 'pakistan', 'Bengali', etc.)
    root_dir = Path.cwd() 
    print(f"Starting processing in: {root_dir}\n")

    # Define the subfolders to look in
    types_to_process = ['HBQ', 'SBQ', 'RBQ']
    
    # Get all subdirectories in the root (e.g., 'pakistan', 'Bengali')
    country_folders = [d for d in root_dir.iterdir() if d.is_dir() and not d.name.startswith('.')]
    print(country_folders)
    country_folders = [
        Path(r'/DATA/rohan_kirti/country/germany'),
        # Path(r"C:\path\to\pakistan"),
        # Add more paths below as needed
        # Path(r"C:\path\to\another_country"),
    ]
    
    if not country_folders:
        print("No country folders (like 'Bengali', 'pakistan') found.")
        print("Please make sure you run this script from the root 'COUNTRY' directory.")
        return

    for country_path in country_folders:
        print(f"Checking Country: {country_path.name}")
        for type_folder in types_to_process:
            # Path to the specific type folder (e.g., .../Bengali/SBQ)
            type_path = country_path / type_folder
            
            if type_path.is_dir():
                # Define the expected CSV file name
                csv_file_name = f"Results_{type_folder}.csv"
                csv_path = type_path / csv_file_name
                
                if csv_path.exists():
                    print(f"-> Found file: {csv_path.relative_to(root_dir)}")
                    process_csv_file(csv_path)
                else:
                    print(f"-> File not found: {csv_path.relative_to(root_dir)}")
            else:
                # This check is useful if a country doesn't have all types (e.g., no 'RBQ')
                # print(f"-> Directory not found: {type_path.relative_to(root_dir)}")
                pass
        print("-" * 20) # Separator



In [13]:
root_dir=Path.cwd()
root_dir.iterdir()
country_folders = [d for d in root_dir.iterdir() if d.is_dir() and not d.name.startswith('.')]
country_folders

[PosixPath('/DATA/rohan_kirti/country/Bengali'),
 PosixPath('/DATA/rohan_kirti/country/sudan'),
 PosixPath('/DATA/rohan_kirti/country/france'),
 PosixPath('/DATA/rohan_kirti/country/china'),
 PosixPath('/DATA/rohan_kirti/country/ethopia'),
 PosixPath('/DATA/rohan_kirti/country/demo'),
 PosixPath('/DATA/rohan_kirti/country/pakistan'),
 PosixPath('/DATA/rohan_kirti/country/indonesia'),
 PosixPath('/DATA/rohan_kirti/country/thailand')]

In [6]:
if __name__ == "__main__":
    main()

Starting processing in: /DATA/rohan_kirti/country

[PosixPath('/DATA/rohan_kirti/country/Bengali'), PosixPath('/DATA/rohan_kirti/country/sudan'), PosixPath('/DATA/rohan_kirti/country/india'), PosixPath('/DATA/rohan_kirti/country/france'), PosixPath('/DATA/rohan_kirti/country/china'), PosixPath('/DATA/rohan_kirti/country/ethopia'), PosixPath('/DATA/rohan_kirti/country/demo'), PosixPath('/DATA/rohan_kirti/country/pakistan'), PosixPath('/DATA/rohan_kirti/country/indonesia'), PosixPath('/DATA/rohan_kirti/country/thailand'), PosixPath('/DATA/rohan_kirti/country/germany')]
Checking Country: germany
-> Found file: germany/HBQ/Results_HBQ.csv
  [SUCCESS] Processed and saved to Filtered_Results_HBQ.xlsx
-> Found file: germany/SBQ/Results_SBQ.csv
  [SUCCESS] Processed and saved to Filtered_Results_SBQ.xlsx
-> Found file: germany/RBQ/Results_RBQ.csv
  [SUCCESS] Processed and saved to Filtered_Results_RBQ.xlsx
--------------------
